In [ ]:
import sys, os
# The shared `processing` module lives in the "Segmentation of tumor cells and vessel
# area in spatial transcriptomic data" step folder. Add it to sys.path so the
# `from processing import ...` call below resolves when this notebook is run from its
# own folder.
sys.path.append(os.path.join("..", "3.Segmentation of tumor cells and vessel area in spatial transcriptomic data"))

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import squidpy as sq

In [ ]:
### 0. example with visulization ###

In [ ]:
directory = '../data/h5ad_batch/'  # Fragmentation complete
file_list = os.listdir(directory)

In [ ]:
## Example code 
from skimage.filters import threshold_otsu
from scipy.ndimage import binary_erosion

cols = adata_.obs['array_col']
rows = adata_.obs['array_row']
weight = adata_.obs['Cancer_Epithelial_filtered']

min_row, max_row = rows.min(), rows.max()
min_col, max_col = cols.min(), cols.max()

n_rows = int(max_row - min_row + 1)
n_cols = int(max_col - min_col + 1)

# Initialize the matrix with NaN (positions with no value remain NaN)
matrix = np.full((n_rows, n_cols), np.nan)

row_index = range(int(min_row), int(max_row) + 1)
col_index = range(int(min_col), int(max_col) + 1)

df = pd.DataFrame(matrix, index=row_index, columns=col_index)


for r, c, w in zip(rows, cols, weight):
    if np.isnan(df.loc[r, c]):
        df.loc[r, c] = w

df = df.fillna(0)

thresh = threshold_otsu(np.array(adata_.obs['Cancer_Epithelial_filtered']))

binary = df > thresh
eroded = binary_erosion(binary.values)
edge_mask = binary & ~eroded

edge_series = edge_mask.stack().rename("edge_cancer").reset_index()
edge_series.columns = ["array_row", "array_col", "edge_cancer"]
adata_obs_updated = adata_.obs.merge(edge_series, on=["array_row", "array_col"], how="left")

# Observations with no matching edge_mask value become NaN, so fill them with False.
adata_obs_updated["edge_cancer"] = adata_obs_updated["edge_cancer"].fillna(False)

# Update adata_.obs with the new DataFrame (or store it in a new variable)
adata_.obs = adata_obs_updated    

adata_.obs['is_cancer'] = np.where(adata_.obs['Cancer_Epithelial_filtered'] > thresh, True, False)
adata_.obs['cancer_area'] = 'others'
adata_.obs.loc[(~adata_.obs['edge_cancer']) & (adata_.obs['is_cancer']),'cancer_area'] = 'cancer'
adata_.obs.loc[(adata_.obs['edge_cancer']) & (adata_.obs['is_cancer']),'cancer_area'] = 'edge'


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 9))
sc.pl.spatial(adata_, color='cancer_area', ax=axes[0], show=False)
sc.pl.spatial(adata_, color='Cancer_Epithelial_filtered', ax=axes[1], show=False,  colorbar_loc=None)


In [ ]:
### 1. Anndata processing and Pseudobulk data generation ###
from processing import find_cell_areas, select_specific_cell

In [ ]:
directory = '../data/h5ad_batch/'  
file_list = os.listdir(directory)
file_list.sort()

adata_list = []
for file_name in file_list:
    adata = sc.read_h5ad(f'{directory}{file_name}')     
    adata.layers['counts'] = adata.X
    sc.pp.normalize_total(adata, target_sum = 1e3)
    sc.pp.log1p(adata)
    adata.layers['log1p'] = adata.X
    adata_list.append(adata)

In [ ]:
# Update the loop to ensure that any COO matrix is converted before further processing
sample_list = [file.split('.')[0] for file in file_list]
flt_adata_list = []

for i, adata in enumerate(adata_list):
    batch_list = adata.obs['batch'].unique().tolist()
    sample_id = sample_list[i]
    for batch in batch_list:
        adata_ = adata[adata.obs['batch'] == batch].copy()
        
        # Process cell areas (this function should work as intended)
        find_cell_areas(adata=adata_)  # Default. cancer_epithelial_filtered
        
        # Convert adata_.X to CSR if it is in COO format to avoid warning issues
        if hasattr(adata_.X, 'format') and adata_.X.format == 'coo':
            adata_.X = adata_.X.tocsr()
        
        adata_1 = select_specific_cell(adata=adata_, cell_type='Cancer_Epithelial_filtered')
        adata_1.obs['sample_id'] = sample_id 
        flt_adata_list.append(adata_1)



In [ ]:
## Pseudobulk expression matrix ##
pseudobulk_data = []
for adata in adata_list_fin:
    X = adata.layers['counts']
    batch = adata.obs['batch'].unique().tolist()[0]
    pseudobulk_counts = X.sum(axis=0)
    pseudobulk_counts = np.array(pseudobulk_counts).flatten()
    total_counts = pseudobulk_counts.sum()
    cpm = (pseudobulk_counts / total_counts) * 1e6
    lognorm = np.log1p(cpm)
    lognorm = lognorm.tolist()
    lognorm.append(batch)
    pseudobulk_data.append(lognorm)

columns_name = adata_list_fin[0].var_names.tolist()
columns_name.append('batch')
pseudobulk_data = pd.DataFrame(pseudobulk_data, columns = columns_name)

In [ ]:
pseudobulk_data.to_csv('../data/pseudobulk_cancer.csv')

In [ ]:
her2_bulk = pseudobulk_data[['ERBB2', 'batch']]

In [ ]:
import os

# Set the target directory path 
directory = "../data/h5ad_cancer/"

## Use only the processed Cancer epithelial regions for the analysis ## 
for adata in adata_list_fin:
    batch = adata.obs['batch'].unique().tolist()[0]
    adata.write_h5ad(f'{directory}{batch}.h5ad')

In [ ]:
##### 2. Compute the spatial autocorrelation index per fragment #####

In [ ]:
## Organize the result dataframe: integrate with clinical information ##

In [ ]:
# Build the per-fragment result table from the fragment batch identifiers.
result_df = pd.DataFrame({'sample_id': [adata.obs['batch'].unique().tolist()[0] for adata in adata_list_fin]})

In [ ]:
clinical = pd.read_csv('../data/clinical.csv', index_col = 0)

In [ ]:
type_list = []
for i in range(len(result_df)):
    clinical_type = clinical['Response to T-Dxd'][clinical['batch'] == result_df.loc[i,'sample_id']].tolist()[0]
    type_list.append(clinical_type)

result_df['clinical_type'] = type_list

In [ ]:
time_list = []
for idx in range(len(result_df)):
    time = clinical[clinical['batch'] == result_df['sample_id'][idx]]['time_point'].iloc[0]
    time_list.append(time)

result_df['time_point'] = time_list

In [ ]:
exp_list = []
for idx in range(len(result_df)):
    her2 = clinical[clinical['batch'] == result_df['sample_id'][idx]]['HER2_type'].iloc[0]
    exp_list.append(her2)

result_df['HER2_type'] = exp_list

In [ ]:
result_df['bulk_her2'] = her2_bulk['ERBB2'].copy()

In [ ]:
### Geary C index of HER2 ####

In [ ]:
import squidpy as sq
import pandas as pd
import numpy as np

results_list = []

for adata in adata_list_fin:
    # Get the batch identifier; assumes there's one unique batch per AnnData object.
    batch = adata.obs['batch'].unique().tolist()[0]
    
    # Compute spatial neighbors and spatial autocorrelation (Moran's I) for the target gene.
    sq.gr.spatial_neighbors(adata, coord_type = 'grid', n_neighs=4)
    sq.gr.spatial_autocorr(
        adata,
        mode='geary',
        genes=['ERBB2'],
        n_perms=100,
        n_jobs=1, 
        show_progress_bar = False
    )
    
    # Extract the Moran's I value for 'ERBB2'
    geary_c = adata.uns['gearyC']['C'].values.item() 
    
    # Get the expression data for 'ERBB2' and compute its mean expression
    target_exp = adata[:, 'ERBB2'].X.toarray()
    mean_exp = np.mean(target_exp).item() 
    
    # Calculate the spand score as defined: negative Moran's I divided by mean expression
    spand_score = -geary_c / mean_exp
    
    # Append the results for this batch to the list
    results_list.append([batch, geary_c, mean_exp, spand_score])
    
# Convert the results list into a DataFrame
df_results_c = pd.DataFrame(results_list, columns=['batch', 'geary_c', 'mean_exp', 'spand_score'])

In [ ]:
### Moran I index of HER2 ####

In [ ]:
import squidpy as sq
import pandas as pd
import numpy as np

results_list = []

for adata in adata_list_fin:
    # Get the batch identifier; assumes there's one unique batch per AnnData object.
    batch = adata.obs['batch'].unique().tolist()[0]
    
    # Compute spatial neighbors and spatial autocorrelation (Moran's I) for the target gene.
    sq.gr.spatial_neighbors(adata, coord_type = 'grid', n_neighs=4)
    sq.gr.spatial_autocorr(
        adata,
        mode='moran',
        genes=['ERBB2'],
        n_perms=100,
        n_jobs=1, 
        show_progress_bar = False
    )
    
    # Extract the Moran's I value for 'ERBB2'
    moran_i = adata.uns['moranI']['I'].values.item() 
    
    # Get the expression data for 'ERBB2' and compute its mean expression
    target_exp = adata[:, 'ERBB2'].X.toarray()
    mean_exp = np.mean(target_exp).item() 
    
    # Calculate the spand score as defined: negative Moran's I divided by mean expression
    spand_score = -moran_i / mean_exp
    
    # Append the results for this batch to the list
    results_list.append([batch, moran_i, mean_exp, spand_score])
    
# Convert the results list into a DataFrame
df_results_i = pd.DataFrame(results_list, columns=['batch', 'moran_i', 'mean_exp', 'spand_score'])

In [ ]:
result_df['geary_c'] = df_results_c['geary_c'].copy()
result_df['moran_i'] = df_results_i['moran_i'].copy()

In [ ]:
result_df.to_csv('../data/results_df_spatial.csv')